In [1]:
import duckdb
print(duckdb.__version__)

1.5.6


In [2]:
import duckdb
from pathlib import Path

BASE = Path(r"C:\Users\Administrator\credit-scoring-lc")
RAW_CSV = BASE / "data" / "raw" / "accepted_2007_to_2018q4.csv" / "accepted_2007_to_2018Q4.csv"
PROC = BASE / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
RAW_PQ = PROC / "accepted_raw.parquet"

con = duckdb.connect()
print(RAW_CSV.exists())

True


In [3]:
con.execute(f"""
COPY (
    SELECT *
    FROM read_csv('{RAW_CSV.as_posix()}',
                  header = true,
                  all_varchar = true,
                  ignore_errors = true)
) TO '{RAW_PQ.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print("done")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done


In [4]:
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM '{RAW_PQ.as_posix()}'
""").df()

,n_rows
0,2260701


In [5]:
con.execute(f"""
SELECT loan_status, COUNT(*) AS n
FROM '{RAW_PQ.as_posix()}'
GROUP BY loan_status
ORDER BY n DESC
""").df()

,loan_status,n
0,Fully Paid,1076751
1,Current,878317
2,Charged Off,268559
3,Late (31-120 days),21467
4,In Grace Period,8436
5,Late (16-30 days),4349
6,Does not meet the credit policy. Status:Fully ...,1988
7,Does not meet the credit policy. Status:Charge...,761
8,Default,40
9,NaN,33


In [6]:
con.execute(f"""
SELECT term, issue_d, emp_length, earliest_cr_line
FROM '{RAW_PQ.as_posix()}'
LIMIT 5
""").df()

,term,issue_d,emp_length,earliest_cr_line
0,36 months,Dec-2015,10+ years,Aug-2003
1,36 months,Dec-2015,10+ years,Dec-1999
2,60 months,Dec-2015,10+ years,Aug-2000
3,60 months,Dec-2015,10+ years,Sep-2008
4,60 months,Dec-2015,3 years,Jun-1998


In [7]:
con.execute(f"""
SELECT year(strptime(issue_d, '%b-%Y')) AS issue_year,
       COUNT(*) AS n_loans,
       SUM(CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END) AS n_default,
       ROUND(AVG(CASE WHEN loan_status = 'Charged Off' THEN 1.0 ELSE 0 END), 4) AS default_rate
FROM '{RAW_PQ.as_posix()}'
WHERE loan_status IN ('Fully Paid', 'Charged Off')
  AND trim(term) = '36 months'
  AND year(strptime(issue_d, '%b-%Y')) BETWEEN 2013 AND 2015
GROUP BY issue_year
ORDER BY issue_year
""").df()

,issue_year,n_loans,n_default,default_rate
0,2013,100422,12378.0,0.1233
1,2014,162570,22315.0,0.1373
2,2015,283026,42132.0,0.1489


In [8]:
con.execute(f"""
SELECT year(strptime(issue_d, '%b-%Y')) AS issue_year,
       ROUND(AVG(CASE WHEN mort_acc IS NULL THEN 1.0 ELSE 0 END), 3) AS mort_acc,
       ROUND(AVG(CASE WHEN tot_cur_bal IS NULL THEN 1.0 ELSE 0 END), 3) AS tot_cur_bal,
       ROUND(AVG(CASE WHEN mths_since_last_delinq IS NULL THEN 1.0 ELSE 0 END), 3) AS mths_since_last_delinq,
       ROUND(AVG(CASE WHEN revol_util IS NULL THEN 1.0 ELSE 0 END), 3) AS revol_util,
       ROUND(AVG(CASE WHEN dti IS NULL THEN 1.0 ELSE 0 END), 3) AS dti
FROM '{RAW_PQ.as_posix()}'
WHERE loan_status IN ('Fully Paid', 'Charged Off')
  AND trim(term) = '36 months'
  AND year(strptime(issue_d, '%b-%Y')) BETWEEN 2013 AND 2015
GROUP BY issue_year
ORDER BY issue_year
""").df()

,issue_year,mort_acc,tot_cur_bal,mths_since_last_delinq,revol_util,dti
0,2013,0.0,0.0,0.565,0.001,0.0
1,2014,0.0,0.0,0.489,0.000,0.0
2,2015,0.0,0.0,0.480,0.000,0.0


In [9]:
con.execute(f"""
CREATE OR REPLACE TABLE loans_clean AS
WITH base AS (
    SELECT *, CAST(strptime(issue_d, '%b-%Y') AS DATE) AS issue_date
    FROM '{RAW_PQ.as_posix()}'
    WHERE loan_status IN ('Fully Paid', 'Charged Off')
      AND trim(term) = '36 months'
)
SELECT
    id,
    issue_date,
    year(issue_date) AS issue_year,
    CASE WHEN year(issue_date) <= 2014 THEN 'train' ELSE 'test' END AS split,
    CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END AS default_flag,
    TRY_CAST(loan_amnt AS DOUBLE) AS loan_amnt,
    purpose,
    initial_list_status,
    CASE WHEN emp_length IS NULL THEN NULL
         WHEN emp_length = '< 1 year' THEN 0
         WHEN emp_length = '10+ years' THEN 10
         ELSE TRY_CAST(regexp_extract(emp_length, '(\\d+)', 1) AS INTEGER)
    END AS emp_length_yrs,
    home_ownership,
    TRY_CAST(annual_inc AS DOUBLE) AS annual_inc,
    verification_status,
    addr_state,
    TRY_CAST(dti AS DOUBLE) AS dti,
    (TRY_CAST(fico_range_low AS DOUBLE) + TRY_CAST(fico_range_high AS DOUBLE)) / 2.0 AS fico,
    TRY_CAST(delinq_2yrs AS DOUBLE) AS delinq_2yrs,
    TRY_CAST(inq_last_6mths AS DOUBLE) AS inq_last_6mths,
    TRY_CAST(mths_since_last_delinq AS DOUBLE) AS mths_since_last_delinq,
    TRY_CAST(mths_since_last_record AS DOUBLE) AS mths_since_last_record,
    TRY_CAST(open_acc AS DOUBLE) AS open_acc,
    TRY_CAST(total_acc AS DOUBLE) AS total_acc,
    TRY_CAST(pub_rec AS DOUBLE) AS pub_rec,
    TRY_CAST(pub_rec_bankruptcies AS DOUBLE) AS pub_rec_bankruptcies,
    TRY_CAST(revol_bal AS DOUBLE) AS revol_bal,
    TRY_CAST(replace(revol_util, '%', '') AS DOUBLE) AS revol_util,
    date_diff('month',
              CAST(strptime(earliest_cr_line, '%b-%Y') AS DATE),
              issue_date) AS credit_hist_months
FROM base
WHERE year(issue_date) BETWEEN 2013 AND 2015
""")
print("loans_clean 建好了")

loans_clean 建好了


In [10]:
con.execute("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT id) AS distinct_ids,
       SUM(CASE WHEN default_flag IS NULL THEN 1 ELSE 0 END) AS null_label,
       SUM(CASE WHEN split = 'train' THEN 1 ELSE 0 END) AS n_train,
       SUM(CASE WHEN split = 'test' THEN 1 ELSE 0 END) AS n_test
FROM loans_clean
""").df()

,total_rows,distinct_ids,null_label,n_train,n_test
0,546018,546018,0.0,262992.0,283026.0


In [11]:
CLEAN_PQ = PROC / "loans_clean.parquet"
con.execute(f"COPY loans_clean TO '{CLEAN_PQ.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(CLEAN_PQ.exists())

True


In [12]:
import pandas as pd

df = pd.read_parquet(CLEAN_PQ)
miss = df.isna().mean().round(3).sort_values(ascending=False)
miss[miss > 0]

mths_since_last_record    0.824
mths_since_last_delinq    0.498
emp_length_yrs            0.062
dtype: float64

In [13]:
sql_dir = BASE / "sql"
sql_dir.mkdir(exist_ok=True)
print(sql_dir.exists())

True


In [14]:
(BASE / ".gitignore").write_text("data/\n*.parquet\n*.csv\n*.gz\n.ipynb_checkpoints/\n")
print((BASE / ".gitignore").exists())

True


In [15]:
sql_text = r"""-- 01_clean.sql
-- Purpose: build the modeling table from LendingClub accepted loans
-- Sample: Fully Paid / Charged Off, 36-month term, issued 2013-2015
-- Split: 2013-2014 = train, 2015 = test. Only application-time variables are kept.

CREATE OR REPLACE TABLE loans_clean AS
WITH base AS (
    SELECT *, CAST(strptime(issue_d, '%b-%Y') AS DATE) AS issue_date
    FROM 'data/processed/accepted_raw.parquet'
    WHERE loan_status IN ('Fully Paid', 'Charged Off')
      AND trim(term) = '36 months'
)
SELECT
    id,
    issue_date,
    year(issue_date) AS issue_year,
    CASE WHEN year(issue_date) <= 2014 THEN 'train' ELSE 'test' END AS split,
    CASE WHEN loan_status = 'Charged Off' THEN 1 ELSE 0 END AS default_flag,
    TRY_CAST(loan_amnt AS DOUBLE) AS loan_amnt,
    purpose,
    initial_list_status,
    CASE WHEN emp_length IS NULL THEN NULL
         WHEN emp_length = '< 1 year' THEN 0
         WHEN emp_length = '10+ years' THEN 10
         ELSE TRY_CAST(regexp_extract(emp_length, '(\d+)', 1) AS INTEGER)
    END AS emp_length_yrs,
    home_ownership,
    TRY_CAST(annual_inc AS DOUBLE) AS annual_inc,
    verification_status,
    addr_state,
    TRY_CAST(dti AS DOUBLE) AS dti,
    (TRY_CAST(fico_range_low AS DOUBLE) + TRY_CAST(fico_range_high AS DOUBLE)) / 2.0 AS fico,
    TRY_CAST(delinq_2yrs AS DOUBLE) AS delinq_2yrs,
    TRY_CAST(inq_last_6mths AS DOUBLE) AS inq_last_6mths,
    TRY_CAST(mths_since_last_delinq AS DOUBLE) AS mths_since_last_delinq,
    TRY_CAST(mths_since_last_record AS DOUBLE) AS mths_since_last_record,
    TRY_CAST(open_acc AS DOUBLE) AS open_acc,
    TRY_CAST(total_acc AS DOUBLE) AS total_acc,
    TRY_CAST(pub_rec AS DOUBLE) AS pub_rec,
    TRY_CAST(pub_rec_bankruptcies AS DOUBLE) AS pub_rec_bankruptcies,
    TRY_CAST(revol_bal AS DOUBLE) AS revol_bal,
    TRY_CAST(replace(revol_util, '%', '') AS DOUBLE) AS revol_util,
    date_diff('month',
              CAST(strptime(earliest_cr_line, '%b-%Y') AS DATE),
              issue_date) AS credit_hist_months
FROM base
WHERE year(issue_date) BETWEEN 2013 AND 2015;
"""
(BASE / "sql" / "01_clean.sql").write_text(sql_text, encoding="utf-8")
print((BASE / "sql" / "01_clean.sql").exists())

True
